In [1]:
import gurobipy as gp
from gurobipy import GRB

# Define the data
suppliers = ['S1', 'S2', 'S3']
consumers = ['C1', 'C2', 'C3']
demand = [30, 40, 30]
capacity = [[10, 20, 30], [15, 25, 10], [20, 15, 25]]
cost = [[4, 6, 8], [7, 2, 1], [6, 3, 2]]

# Create a model
model = gp.Model("Transportation")

# Create variables
x = {}
for i in range(len(suppliers)):
    for j in range(len(consumers)):
        x[i, j] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_{suppliers[i]}_{consumers[j]}")

# Objective function
model.setObjective(gp.quicksum(cost[i][j] * x[i, j] for i in range(len(suppliers)) for j in range(len(consumers))), GRB.MINIMIZE)

# Demand constraints
for j in range(len(consumers)):
    model.addConstr(gp.quicksum(x[i, j] for i in range(len(suppliers))) >= demand[j], name=f"demand_{consumers[j]}")

# Supply constraints
for i in range(len(suppliers)):
    model.addConstr(gp.quicksum(x[i, j] for j in range(len(consumers))) >= 0, name=f"supply_{suppliers[i]}")

# Capacity constraints
for i in range(len(suppliers)):
    for j in range(len(consumers)):
        model.addConstr(x[i, j] <= capacity[i][j], name=f"capacity_{suppliers[i]}_{consumers[j]}")

# Flow conservation constraints for demand nodes
for j in range(len(consumers)):
    model.addConstr(gp.quicksum(x[i, j] for i in range(len(suppliers))) == gp.quicksum(x[k, j] for k in range(len(suppliers))), name=f"flow_conservation_demand_{consumers[j]}")

# Solve the model
model.optimize()

# Print the solution
if model.status == GRB.OPTIMAL:
    print("Optimal solution found:")
    for i in range(len(suppliers)):
        for j in range(len(consumers)):
            if x[i, j].x > 0:
                print(f"Ship {x[i, j].x} units from {suppliers[i]} to {consumers[j]}")
    print("Total cost:", model.objVal)
else:
    print("No solution found")


Set parameter Username
Academic license - for non-commercial use only - expires 2024-12-20
Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 18 rows, 9 columns and 27 nonzeros
Model fingerprint: 0x86610fb7
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 8e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 4e+01]
Presolve removed 17 rows and 6 columns
Presolve time: 0.00s
Presolved: 1 rows, 3 columns, 3 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.5500000e+02   1.500000e+01   0.000000e+00      0s
Extra simplex iterations after uncrush: 2
       3    3.0500000e+02   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.01 seconds (0.00 work units)
Optimal objective  3.0

In [27]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("WarehouseProblem")

# Parameters
slack_nodes = ['Slack1', 'Slack2', 'Slack3']
supply_nodes = ['S1', 'S2', 'S3']  # Supply nodes
demand_nodes = ['D1', 'D2', 'D3']  # Demand nodes
edges = [('S1', 'D1'), ('S1', 'D2'), ('S2', 'D2'), ('S2', 'D3'), ('S3', 'D1') ,('Slack1', 'S1'), ('Slack2', 'S2'), ('Slack3', 'S3')]  # Edges
network_edges = [('S1', 'D1'), ('S1', 'D2'), ('S2', 'D2'), ('S2', 'D3'), ('S3', 'D1')]
slack_edges = [('Slack1', 'S1'), ('Slack2', 'S2'), ('Slack3', 'S3')]
capacities = {'S1D1': 20, 'S1D2': 30, 'S2D2': 25, 'S2D3': 40, 'S3D1': 35}  # Capacities

# Cost matrix (replace this with your actual cost values)
costs = {'S1D1': 5, 'S1D2': 8, 'S2D2': 6, 'S2D3': 10, 'S3D1': 7}

# Demand values for each node
demand = {'D1': 15, 'D2': 20, 'D3': 25}

# Supply values for each node
supply = {'S1': 10, 'S2': 10, 'S3': 15}

# Decision variables
x = {}
for edge in edges:
    x[edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_{edge[0]}_{edge[1]}")

# Objective function (minimize total transportation cost)
model.setObjective(gp.quicksum(x[edge] * costs[f"{edge[0]}{edge[1]}"] for edge in network_edges), GRB.MINIMIZE)

# Constraints

# Slack constraints
for node in slack_nodes:
    model.addConstr(gp.quicksum(x[edge] for edge in slack_edges if edge[0] == node) >= 0, f"slack_{node}")

# Supply constraints
for node in supply_nodes:
    model.addConstr(gp.quicksum(x[edge] for edge in network_edges if edge[0] == node)
                    -gp.quicksum(x[edge] for edge in network_edges if edge[1] == node)
                    == supply[node], f"supply_{node}")

# Demand constraints
for node in demand_nodes:
    model.addConstr(gp.quicksum(x[edge] for edge in network_edges if edge[1] == node) == demand[node], f"demand_{node}")

# Capacity constraints
for edge in network_edges:
    model.addConstr(x[edge] <= capacities[f"{edge[0]}{edge[1]}"], f"capacity_{edge[0]}_{edge[1]}")

# Additional constraint: Ensure supply nodes are not balanced
#model.addConstr(gp.quicksum(supply[node] for node in supply_nodes) != gp.quicksum(demand[node] for node in demand_nodes), "not_balanced")

# Optimize the model
model.optimize()

# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
    for edge in edges:
        print(f"{edge}: {x[edge].x}")
    print(f"Total cost: {model.objVal}")
else:
    print("No optimal solution found.")


Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 14 rows, 8 columns and 18 nonzeros
Model fingerprint: 0x75a24537
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 1e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 4e+01]
Presolve removed 11 rows and 6 columns
Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model
No optimal solution found.


In [28]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model
model = gp.Model("WarehouseProblem")

# Parameters
supply_nodes = ['S1', 'S2', 'S3']  # Supply nodes
demand_nodes = ['D1', 'D2', 'D3']  # Demand nodes
edges = [('S1', 'D1'), ('S1', 'D2'), ('S2', 'D2'), ('S2', 'D3'), ('S3', 'D1')]  # Edges
capacities = {'S1D1': 20, 'S1D2': 30, 'S2D2': 25, 'S2D3': 40, 'S3D1': 35}  # Capacities

# Cost matrix (replace this with your actual cost values)
costs = {'S1D1': 5, 'S1D2': 8, 'S2D2': 6, 'S2D3': 10, 'S3D1': 7}

# Demand values for each node
demand = {'D1': 15, 'D2': 20, 'D3': 25}

# Supply values for each node
supply = {'S1': 30, 'S2': 40, 'S3': 35}

# Decision variables
x = {}
for edge in edges:
    x[edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_{edge[0]}_{edge[1]}")

# Objective function (minimize total transportation cost)
model.setObjective(gp.quicksum(x[edge] * costs[f"{edge[0]}{edge[1]}"] for edge in edges), GRB.MINIMIZE)

# Constraints

# Supply constraints
for node in supply_nodes:
    model.addConstr(gp.quicksum(x[edge] for edge in edges if edge[0] == node) <= supply[node], f"supply_{node}")

# Demand constraints
for node in demand_nodes:
    model.addConstr(gp.quicksum(x[edge] for edge in edges if edge[1] == node) == demand[node], f"demand_{node}")

# Capacity constraints
for edge in edges:
    model.addConstr(x[edge] <= capacities[f"{edge[0]}{edge[1]}"], f"capacity_{edge[0]}_{edge[1]}")

# Optimize the model
model.optimize()

# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
    for edge in edges:
        print(f"{edge}: {x[edge].x}")
    print(f"Total cost: {model.objVal}")
else:
    print("No optimal solution found.")


Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 11 rows, 5 columns and 15 nonzeros
Model fingerprint: 0xdf553b5d
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+00, 1e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 4e+01]
Presolve removed 11 rows and 5 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.5500000e+02   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  4.550000000e+02
Optimal solution found!
('S1', 'D1'): 15.0
('S1', 'D2'): 5.0
('S2', 'D2'): 15.0
('S2', 'D3'): 25.0
('S3', 'D1'): 0.0
Total cost: 455.0


In [29]:
# Copyright 2023, Gurobi Optimization, LLC

# Solve a multi-commodity flow problem.  Two products ('Pencils' and 'Pens')
# are produced in 2 cities ('Detroit' and 'Denver') and must be sent to
# warehouses in 3 cities ('Boston', 'New York', and 'Seattle') to
# satisfy supply/demand ('inflow[h,i]').
#
# Flows on the transportation network must respect arc capacity constraints
# ('capacity[i,j]'). The objective is to minimize the sum of the arc
# transportation costs ('cost[i,j]').

import gurobipy as gp
from gurobipy import GRB

# Base data
commodities = ["Pencils", "Pens"]
nodes = ["Detroit", "Denver", "Boston", "New York", "Seattle"]

arcs, capacity = gp.multidict(
    {
        ("Detroit", "Boston"): 100,
        ("Detroit", "New York"): 80,
        ("Detroit", "Seattle"): 120,
        ("Denver", "Boston"): 120,
        ("Denver", "New York"): 120,
        ("Denver", "Seattle"): 120,
    }
)

# Cost for triplets commodity-source-destination
cost = {
    ("Pencils", "Detroit", "Boston"): 10,
    ("Pencils", "Detroit", "New York"): 20,
    ("Pencils", "Detroit", "Seattle"): 60,
    ("Pencils", "Denver", "Boston"): 40,
    ("Pencils", "Denver", "New York"): 40,
    ("Pencils", "Denver", "Seattle"): 30,
    ("Pens", "Detroit", "Boston"): 20,
    ("Pens", "Detroit", "New York"): 20,
    ("Pens", "Detroit", "Seattle"): 80,
    ("Pens", "Denver", "Boston"): 60,
    ("Pens", "Denver", "New York"): 70,
    ("Pens", "Denver", "Seattle"): 30,
}

# Supply (> 0) and demand (< 0) for pairs of commodity-city
inflow = {
    ("Pencils", "Detroit"): 50,
    ("Pencils", "Denver"): 60,
    ("Pencils", "Boston"): -50,
    ("Pencils", "New York"): -50,
    ("Pencils", "Seattle"): -10,
    ("Pens", "Detroit"): 60,
    ("Pens", "Denver"): 40,
    ("Pens", "Boston"): -40,
    ("Pens", "New York"): -30,
    ("Pens", "Seattle"): -30,
}

# Create optimization model
m = gp.Model("netflow")

# Create variables
flow = m.addVars(commodities, arcs, obj=cost, name="flow")

# Arc-capacity constraints
m.addConstrs((flow.sum("*", i, j) <= capacity[i, j] for i, j in arcs), "cap")

# Equivalent version using Python looping
# for i, j in arcs:
#   m.addConstr(sum(flow[h, i, j] for h in commodities) <= capacity[i, j],
#               "cap[%s, %s]" % (i, j))


# Flow-conservation constraints
m.addConstrs(
    (
        flow.sum(h, "*", j) + inflow[h, j] == flow.sum(h, j, "*")
        for h in commodities
        for j in nodes
    ),
    "node",
)

# Alternate version:
# m.addConstrs(
#   (gp.quicksum(flow[h, i, j] for i, j in arcs.select('*', j)) + inflow[h, j] ==
#     gp.quicksum(flow[h, j, k] for j, k in arcs.select(j, '*'))
#     for h in commodities for j in nodes), "node")

# Compute optimal solution
m.optimize()

# Print solution
if m.Status == GRB.OPTIMAL:
    solution = m.getAttr("X", flow)
    for h in commodities:
        print(f"\nOptimal flows for {h}:")
        for i, j in arcs:
            if solution[h, i, j] > 0:
                print(f"{i} -> {j}: {solution[h, i, j]:g}")

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 12 columns and 36 nonzeros
Model fingerprint: 0xc43e5943
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 8e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]
Presolve removed 16 rows and 12 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.5000000e+03   0.000000e+00   2.000000e+01      0s
Extra simplex iterations after uncrush: 1
       1    5.5000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.02 seconds (0.00 work units)
Optimal objective  5.500000000e+03

Optimal flows for Pencils:
Detroit -> Boston: 50
Denver -> New York: 50
Denver